<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/Rag_MultiplePDF_MedicalDoc_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai langchain-community pypdf langchain-text-splitters chromadb gradio

import os
import gradio as gr
from google.colab import userdata
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["OPENAI_API_KEY"] = #Use your OPENAI_API_KEY using userdata.get("OPENAI_API_KEY")

# Global chain
chain = None

# ══════════════════════════════════════════════════════════════════════════════
# GUARDRAIL
# ══════════════════════════════════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a document assistant. Follow these rules strictly:
1. Answer ONLY using the context provided below.
2. If the answer is not found in the context, respond with exactly: "I don't have enough information in the document to answer this."
3. Do not use any outside knowledge.
4. Do not make up facts, names, or figures.

Context:
{context}"""


def process_pdfs(files):
    global chain

    if not files:
        return "Please upload at least one PDF."

    # Load & split all files into one combined chunk list
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
    all_chunks = []

    for file in files:
        all_chunks.extend(splitter.split_documents(PyPDFLoader(file.name).load()))

    vectorstore = Chroma.from_documents(
        all_chunks,
        OpenAIEmbeddings(),
        persist_directory="./chroma_db"
    )
    chunk_count = vectorstore._collection.count()

    retriever = vectorstore.as_retriever()

    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        ("human", "{question}")
    ])

    chain = (
        {"context": retriever, "question": lambda x: x}
        | prompt
        | ChatOpenAI(model="gpt-4o-mini", temperature=0)
        | StrOutputParser()
    )

    file_names = "\n".join([f"  • {os.path.basename(f.name)}" for f in files])
    return f"{len(files)} PDF(s) indexed successfully!\n\n Files:\n{file_names}\n\n Total chunks stored: {chunk_count} \n\nYou can now ask questions below."


def ask_question(question):
    global chain

    if chain is None:
        return " No PDF loaded yet. Please upload and process a PDF first."

    return chain.invoke(question)


# ── Gradio UI ──────────────────────────────────────────────────────────────────
with gr.Blocks(title=" PDF RAG Assistant", theme=gr.themes.Soft()) as demo:

    gr.Markdown("##  PDF RAG Assistant\nUpload one or more PDFs, index them, then ask questions powered by GPT-4o-mini + ChromaDB.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Step 1 — Upload & Index PDFs")
            pdf_input    = gr.File(label="Upload PDFs", file_types=[".pdf"], file_count="multiple")
            index_btn    = gr.Button(" Index PDFs", variant="primary")
            index_status = gr.Textbox(label="Status", lines=6, interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### Step 2 — Ask a Question")
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g. Summarize the Medical policy document",
                lines=2
            )
            ask_btn = gr.Button(" Ask", variant="primary")
            answer  = gr.Textbox(label="Answer", lines=10, interactive=False)

    index_btn.click(fn=process_pdfs, inputs=pdf_input,      outputs=index_status)
    ask_btn.click(  fn=ask_question, inputs=question_input, outputs=answer)
    question_input.submit(fn=ask_question, inputs=question_input, outputs=answer)

demo.launch(share=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69

/tmp/ipykernel_6983/3436831712.py:79: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="📄 PDF RAG Assistant", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://07943dcf705e7d1fd9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
